# Whisper Large v3 Evaluation\n\nThis notebook evaluates the OpenAI Whisper Large v3 model on a dataset of audio segments.\nIt uses the direct batch inference approach with the shared evaluation runner.

In [ ]:
#@title Install packages
import builtins

# The Magic Hack: Create a dummy class and inject it into Python's builtins 
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass

builtins.PeftConfigLike = DummyPeftConfig

import os
import json
import sys
import torch
from transformers import pipeline
from google.cloud import storage

# Add project root to path
sys.path.append("/app")

# Import common utils
from colabs.common.gcs_utils import download_jsonl_manifest, upload_inference_results
from colabs.common.eval_runner import run_inference_pipeline
from colabs.common.audio_utils import preprocess_audio_for_model

# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# --- Configuration ---
MODEL_NAME = "openai/whisper-large-v3-turbo" # Alternate /whisper-large-v3
SELECTED_MODEL_KEY = "whisper_v3_turbo" 

GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>"
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"

# NOTE: If using full large-v3 (non-turbo), reduce batch size to avoid OOM (e.g., 1 or 2)
BATCH_SIZE = 4
LIMIT = 10

In [ ]:
#@title Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Whisper model on {device}...")

pipe = pipeline(
    "automatic-speech-recognition",
    model=MODEL_NAME,
    device=device,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

In [ ]:
#@title Define helper functions for evaluation runner

def prompt_formatter(entry, local_path):
    """For Whisper pipeline, we just need to pass the file path."""
    return local_path

def whisper_inference(model, prompts):
    """Runs inference using the pipeline object with shared audio preprocessing."""
    outputs = []
    for audio_path in prompts:
        clean_path = audio_path + "_clean.wav"
        
        try:
            # Use the shared function from audio_utils!
            success = preprocess_audio_for_model(audio_path, clean_path)
            
            if not success:
                logger.error(f"Preprocessing failed for {audio_path}")
                continue
                
            # Run model (model is the pipe object)
            # Whisper pipeline accepts file path directly
            # Added low temperature and no_repeat_ngram_size to prevent looping
            out = model(
                clean_path,
                generate_kwargs={
                    "temperature": 0.0,
                    "no_repeat_ngram_size": 3, 
                }
            )
            outputs.append(out)
            
        finally:
            # Clean up the temporary WAV file
            if os.path.exists(clean_path):
                os.remove(clean_path)
                
    return outputs

def result_decoder(ans, model):
    """Extracts the transcription from Whisper's output."""
    return ans["text"]

In [ ]:
#@title Run Evaluation

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Run the generic batch evaluation
results_list = run_inference_pipeline(
    model=pipe,
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=whisper_inference,
    decode_fn=result_decoder,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=SELECTED_MODEL_KEY,
    batch_size=BATCH_SIZE,
    limit=LIMIT
)

In [ ]:

# Upload results directly to GCS from memory
gcs_uri = upload_inference_results(
    storage_client, 
    GCS_BUCKET, 
    PROJECT_NAME, 
    SELECTED_MODEL_KEY, 
    EXPERIMENT_NAME, 
    results_list
)